In [1]:
import pandas as pd
import sys, os
sys.path.insert(0, '../etl')
from db_connection import get_engine

engine = get_engine()


In [2]:
# 1. Revenue by month — should show seasonal pattern
monthly = pd.read_sql("""
    SELECT d.year, d.month, d.month_name,
           COUNT(DISTINCT f.invoice_id)   AS orders,
           ROUND(SUM(f.revenue)::numeric, 2) AS revenue
    FROM fact_orders f
    JOIN dim_date d ON f.date_id = d.date_id
    WHERE f.is_cancelled = FALSE
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""", engine)
print(monthly.to_string(index=False))

 year  month month_name  orders    revenue
 2009     12  December     1671  798514.66
 2010      1  January      1089  621435.35
 2010      2  February     1189  538115.62
 2010      3  March        1647  762105.98
 2010      4  April        1437  646791.23
 2010      5  May          1484  643742.03
 2010      6  June         1615  697766.79
 2010      7  July         1509  633738.19
 2010      8  August       1402  674687.72
 2010      9  September    1789  869693.12
 2010     10  October      2244 1095285.98
 2010     11  November     2722 1437772.49
 2010     12  December     1552  789782.24
 2011      1  January      1081  670580.24
 2011      2  February     1093  508014.87
 2011      3  March        1441  690803.27
 2011      4  April        1236  515807.99
 2011      5  May          1668  740389.00
 2011      6  June         1525  738108.98
 2011      7  July         1452  688769.34
 2011      8  August       1341  735678.56
 2011      9  September    1819 1029220.38
 2011     1

In [3]:
# 2. Top 10 products by revenue
top_products = pd.read_sql("""
    SELECT p.description, p.stock_code,
           SUM(f.quantity)                   AS units_sold,
           ROUND(SUM(f.revenue)::numeric, 2) AS revenue
    FROM fact_orders f
    JOIN dim_product p ON f.stock_code = p.stock_code
    WHERE f.is_cancelled = FALSE
    GROUP BY p.description, p.stock_code
    ORDER BY revenue DESC LIMIT 10
""", engine)
print(top_products)

                          description stock_code  units_sold    revenue
0            REGENCY CAKESTAND 3 TIER      22423       26478  330590.32
1  WHITE HANGING HEART T-LIGHT HOLDER     85123A       94203  257724.71
2             JUMBO BAG RED RETROSPOT     85099B       96757  180569.34
3         PAPER CRAFT , LITTLE BIRDIE      23843       80995  168469.60
4                       PARTY BUNTING      47566       28200  148318.28
5       ASSORTED COLOUR BIRD ORNAMENT      84879       80082  129324.49
6     PAPER CHAIN KIT 50'S CHRISTMAS       22086       35084  117760.29
7      MEDIUM CERAMIC TOP STORAGE JAR      23166       78033   81700.92
8                       CHILLI LIGHTS      79321       15841   80540.88
9                SMALL POPCORN HOLDER      22197       88499   79520.20


In [4]:
# 3. Customer segmentation by order count
seg = pd.read_sql("""
    SELECT
        CASE
            WHEN total_orders = 1 THEN '1 order'
            WHEN total_orders BETWEEN 2 AND 5 THEN '2–5 orders'
            WHEN total_orders BETWEEN 6 AND 20 THEN '6–20 orders'
            ELSE '20+ orders'
        END AS segment,
        COUNT(*) AS customers
    FROM dim_customer
    WHERE is_guest = FALSE
    GROUP BY 1 ORDER BY customers DESC
""", engine)
print(seg)

       segment  customers
0   2–5 orders       2368
1  6–20 orders       1670
2      1 order       1447
3   20+ orders        410


In [5]:
# 4. ETL audit trail
log = pd.read_sql("SELECT * FROM etl_log ORDER BY run_timestamp DESC LIMIT 5", engine)
print(log)

   log_id              run_timestamp  rows_extracted  rows_loaded  \
0       1 2026-05-22 10:35:46.875429         1067371      1021733   

   rows_skipped   status              notes  
0         45638  SUCCESS  Completed in 174s  
